In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
import statsmodels.formula.api as smf
import matplotlib.ticker as ticker
import numpy as np
import pingouin
import scipy.stats as stats
from scipy.stats import ttest_rel


In [ ]:
# Function Definitions

def fix_name(row):
    """Resolve a model name for step checkpoints.

    For checkpoints identified by a path containing '@step', returns the last
    path component (e.g. 'EleutherAI/pythia-14m@step_512' -> 'pythia-14m@step_512').
    Otherwise returns the pre-existing 'name' field unchanged.
    """
    if "@step" in row["checkpoint"]:
        return row["checkpoint"].split("/")[-1]
    return row["name"]


def compute_delta_spearman(df, index_cols, pivot_col="standardization"):
    """Compute the per-row delta in Spearman correlation between two conditions.

    Pivots `df` so that each value of `pivot_col` becomes a column, then takes
    the column-wise diff so the last column holds (condition_B - condition_A).
    Returns a tidy DataFrame with the index columns plus a 'delta_spearman' column.

    Parameters
    ----------
    df : DataFrame
        Must contain 'spearman' and `pivot_col` columns alongside `index_cols`.
    index_cols : list[str]
        Columns to use as the pivot index (e.g. ["name", "layer", "Data Scale"]).
    pivot_col : str
        Column whose two values define the conditions to diff (default: 'standardization').
    """
    return (
        df
        .pivot_table(index=index_cols, columns=pivot_col, values="spearman")
        .diff(axis=1)
        .iloc[:, -1]
        .reset_index(name="delta_spearman")
    )


def merge_with_anisotropy(delta_df, anisotropy_df):
    """Join a delta-spearman DataFrame with the anisotropy measurements.

    Normalises both key columns to lowercase before merging on (name, layer)
    to handle inconsistent capitalisation across result files.
    The redundant 'run' column from `anisotropy_df` is dropped from the result.

    Parameters
    ----------
    delta_df : DataFrame
        Must contain 'name' and 'layer' columns.
    anisotropy_df : DataFrame
        Must contain 'run', 'layer', and 'anisotropy' columns.
    """
    return (
        pd.merge(
            delta_df.assign(name=delta_df["name"].str.lower()),
            anisotropy_df.assign(run=anisotropy_df["run"].str.lower()),
            left_on=["name", "layer", "step"],
            right_on=["run", "layer", "step"],
        )
        .drop(columns=["run"])
    )


def compute_checkpoint_delta_merged(checkpoint_df, anisotropy_df):
    """Full pipeline: compute delta-spearman and merge with anisotropy for checkpoints.

    Combines compute_delta_spearman -> fix_name -> merge_with_anisotropy into a
    single call for checkpoint-level DataFrames that have 'checkpoint' and 'step'
    columns. Returns the merged DataFrame sorted by step.

    Parameters
    ----------
    checkpoint_df : DataFrame
        Checkpoint rows with 'checkpoint', 'layer', 'name', 'step', and
        'standardization' columns.
    anisotropy_df : DataFrame
        Anisotropy measurements with 'run', 'layer', and 'anisotropy' columns.
    """
    delta_df = compute_delta_spearman(
        checkpoint_df, ["checkpoint", "layer", "name", "step"]
    )
    delta_df["name"] = delta_df.apply(fix_name, axis=1)
    return merge_with_anisotropy(delta_df, anisotropy_df).sort_values("step")


def plot_anisotropy_delta_scatter(df, title, hue="name", style=None):
    """Scatter plot of anisotropy vs. delta-spearman with an OLS regression line.

    Parameters
    ----------
    df : DataFrame
        Must contain 'anisotropy' and 'delta_spearman' columns.
    title : str
        Plot title.
    hue : str
        Column to use for point colour (default: 'name').
    style : str or None
        Column to use for point marker style (default: None).
    """
    sns.scatterplot(data=df, x="anisotropy", y="delta_spearman", hue=hue, style=style)
    plt.title(title)
    sns.regplot(data=df, x="anisotropy", y="delta_spearman", scatter=False,
                color="gray", line_kws={"linestyle": "--"})
    plt.xlabel("Anisotropy")
    plt.ylabel("Delta Spearman Correlation")
    plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
    plt.show()


def report_spearman(df, label="", indep_var="anisotropy", dep_var="delta_spearman"):
    """Print the Spearman correlation between anisotropy and delta-spearman.

    Parameters
    ----------
    df : DataFrame
        Must contain 'anisotropy' and 'delta_spearman' columns.
    label : str
        Optional suffix appended to the printed line to identify the model/subset.

    Returns
    -------
    (r, p) : tuple[float, float]
        Spearman correlation coefficient and two-tailed p-value.
    """
    r, p = spearmanr(df[indep_var], df[dep_var])
    suffix = f" {label}" if label else ""
    print(f"Spearman r between {indep_var} and {dep_var}{suffix}: {r:.4f}, p={p:.4f}")
    return r, p


def plot_spearman_by_tokens_seen(df, title):
    """Line plot of average Spearman correlation vs. tokens seen (log scale).

    Draws one line per standardization condition. The x-axis is log-scaled and
    formatted as 'XM' / 'XB' for readability.

    Parameters
    ----------
    df : DataFrame
        Must contain 'tokens_seen', 'spearman', and 'standardization' columns.
    title : str
        Plot title.
    """
    plt.figure(figsize=(16, 5))
    sns.lineplot(data=df, x="tokens_seen", y="spearman", hue="standardization")
    plt.xscale("log")
    plt.gca().xaxis.set_major_formatter(ticker.FuncFormatter(
        lambda x, _: f"{x/1e9:.0f}B" if x >= 1e9 else f"{x/1e6:.0f}M"
    ))
    plt.title(title, fontsize=16)
    plt.xlabel("Tokens Seen (log scale)", fontsize=14)
    plt.ylabel("Average Spearman Correlation", fontsize=14)
    plt.legend(title="Standardization Condition", fontsize=12)
    plt.tick_params(axis="both", labelsize=12)
    plt.tight_layout()
    plt.show()


def plot_delta_spearman_by_tokens_seen(df, title):
    """Line plot of delta Spearman correlation vs. tokens seen (log scale).

    Draws one line incorporating the difference between standardizations. 
    The x-axis is log-scaled and formatted as 'XM' / 'XB' for readability.

    Parameters
    ----------
    df : DataFrame
        Must contain 'tokens_seen' and 'delta_spearman' columns.
    title : str
        Plot title.
    """
    plt.figure(figsize=(16, 5))
    sns.lineplot(data=df, x="tokens_seen", y="delta_spearman")
    plt.xscale("log")
    plt.gca().xaxis.set_major_formatter(ticker.FuncFormatter(
        lambda x, _: f"{x/1e9:.0f}B" if x >= 1e9 else f"{x/1e6:.0f}M"
    ))
    # Plot a solid line at y=0 to indicate no difference between standardizations
    plt.axhline(0, color="black", linestyle="--")

    plt.title(title, fontsize=24)
    plt.xlabel("Tokens Seen (log scale)", fontsize=20)
    plt.ylabel("Average Δρ", fontsize=20)
    plt.tick_params(axis="both", labelsize=18)

    plt.tight_layout()
    plt.show()


def tokens_seen(model, step):
    """
    Calculate the number of tokens seen based on the model name and training step.
    This function uses hardcoded batch sizes and token counts for Pythia and Gemstone.
    Do not attempt to generalize beyond these specific models without updating the logic.

    Parameters
    ----------
    model : str
        The name of the model, used to determine which token calculation to apply.
    step : int
        The training step number, used to calculate total tokens seen.
    """
    if "pythia" in model.lower():
        batch_size=2_097_152
        return step * batch_size
    elif "gemstone" in model.lower():
        return step * 2_000_000_000 // 477

def add_tokens_seen_estimates(df):
    """
    For mediation analysis, we need a continuous variable for data size. The BabyLM-scale models use word count, 
    while the Pythia and Gemstone models use token count. We approximate 100 tokens ~= 75 words, based on 
    https://platform.openai.com/tokenizer.
    
    Parameters
    ----------
    df : pandas.DataFrame
        The DataFrame to which the tokens seen estimates will be added.

    Modifies the DataFrame in place, adding a new column 'alt_tokens_seen' with estimated token counts.
    """
    # Add tokens_seen estimates to df.
    tokens_seen_map = {"10M": int(10_000_000 * 100 / 75), "100M": int(100_000_000 * 100 / 75)} # Assumes 75 words ~= 100 tokens.
    df["alt_tokens_seen"] = None
    for i in range(len(df)):
        # If tokens seen is None, estimate it based on the data size using the tokens_seen_map.
        if pd.isna(df.loc[i, "tokens_seen"]):
            data_size = df.loc[i, "data_size"]
            if data_size == "Huge":
                model_name = df.loc[i, "name"]
                if "pythia" in model_name.lower(): # The final steps of Pythia don't have a step number.
                    alt_tokens_seen = 300_000_000_000 # Pythia models are trained on 300B tokens.
                else: # Apply the tokens_seen function to estimate tokens seen for the Huge models based on their name and step.
                    step = df.loc[i, "step"]
                    alt_tokens_seen = tokens_seen(model_name, step)
                    if not alt_tokens_seen:
                        print(f"Warning: Could not estimate tokens seen for model {model_name} at step {step}. Setting to NaN.")
                        alt_tokens_seen = np.nan
                df.loc[i, "alt_tokens_seen"] = alt_tokens_seen
            else:
                df.loc[i, "alt_tokens_seen"] = tokens_seen_map.get(data_size, np.nan)
        else:
            df.loc[i, "alt_tokens_seen"] = df.loc[i, "tokens_seen"]


# Loading all data

In [ ]:
data_dir = "../data"
simlex_results_df = pd.read_csv(f"{data_dir}/simlex_results.csv")
anisotropy_df = pd.read_csv(f"{data_dir}/anisotropy.csv")
dim_reduction_df = pd.read_csv(f"{data_dir}/dim_reduction.csv")
isolated_vs_contextualized_df = pd.read_csv(f"{data_dir}/isolated_vs_contextualized.csv")

## Set Up Across-Model Dataframes
This section sets up analysis of one checkpoint per model. In the case of Pythia and Gemstone models, where we have results from many checkpoints, we use the results from the final checkpoint, which has trained on the full corpus.

In [ ]:
# Get the dataframes for this section.
# For all models except Gemstone and Pythia, the final checkpoint is the only one we have, and it does not include a step number.
# For Pythia, the final checkpoint is the one that does not include a step number.
# For Gemstone, the last checkpoint is the one with the maximum step number.
max_gemstone_step = simlex_results_df[simlex_results_df["name"].str.contains("Gemstone")]["step"].max()

last_gemstone_checkpoint = simlex_results_df[(simlex_results_df["name"].str.contains("Gemstone")) & (simlex_results_df["step"] == max_gemstone_step)]
final_checkpoints = simlex_results_df[simlex_results_df["step"].isnull()] # Everything except Gemstone.
simlex_results_across_model_df = pd.concat([last_gemstone_checkpoint, final_checkpoints], ignore_index=True)

last_gemstone_anisotropy = anisotropy_df[(anisotropy_df["name"].str.contains("Gemstone")) & (anisotropy_df["step"] == max_gemstone_step)]
final_checkpoints_anisotropy = anisotropy_df[anisotropy_df["step"].isnull()]
anisotropy_across_model_df = pd.concat([last_gemstone_anisotropy, final_checkpoints_anisotropy], ignore_index=True)

last_gemstone_dim_reduction_df = dim_reduction_df[(dim_reduction_df["name"].str.contains("Gemstone")) & (dim_reduction_df["step"] == max_gemstone_step)]
final_checkpoints_dim_reduction_df = dim_reduction_df[dim_reduction_df["step"].isnull()]
dim_reduction_across_model_df = pd.concat([last_gemstone_dim_reduction_df, final_checkpoints_dim_reduction_df], ignore_index=True)


Verify all expected rows are there:

In [ ]:
simlex_results_across_model_df[["data_size", "standardization"]].value_counts()

In [ ]:
anisotropy_across_model_df["data_size"].value_counts()

In [ ]:
dim_reduction_across_model_df[["data_size", "k"]].value_counts()

## Set up within-model dataframes
This section sets up multiple checkpoints for the same models, enabling us to see changes in the standardization benefits, anisotropy, and rogue dimensionality within the same model over time.

In [ ]:
# Getting all the dataframes for this section.
simlex_results_within_model_df = simlex_results_df[~simlex_results_df["step"].isnull()]
anisotropy_within_model_df = anisotropy_df[~anisotropy_df["step"].isnull()]
dim_reduction_within_model_df = dim_reduction_df[~dim_reduction_df["step"].isnull()]

# Add the tokens_seen column to these dfs, since they should all have a step value.
simlex_results_within_model_df["tokens_seen"] = simlex_results_within_model_df.apply(lambda row: tokens_seen(row["name"], row["step"]), axis=1)
anisotropy_within_model_df["tokens_seen"] = anisotropy_within_model_df.apply(lambda row: tokens_seen(row["name"], row["step"]), axis=1)
dim_reduction_within_model_df["tokens_seen"] = dim_reduction_within_model_df.apply(lambda row: tokens_seen(row["name"], row["step"]), axis=1)

# And specific to each model:
simlex_results_p14_df = simlex_results_within_model_df[simlex_results_within_model_df["name"] == "Pythia-14M"]
anisotropy_p14_df = anisotropy_within_model_df[anisotropy_within_model_df["name"] == "Pythia-14M"]
dim_reduction_p14_df = dim_reduction_within_model_df[dim_reduction_within_model_df["name"] == "Pythia-14M"]

simlex_results_p70_df = simlex_results_within_model_df[simlex_results_within_model_df["name"] == "Pythia-70M"]
anisotropy_p70_df = anisotropy_within_model_df[anisotropy_within_model_df["name"] == "Pythia-70M"]
dim_reduction_p70_df = dim_reduction_within_model_df[dim_reduction_within_model_df["name"] == "Pythia-70M"]

simlex_results_g256_df = simlex_results_within_model_df[simlex_results_within_model_df["name"] == "Gemstone-256x23"]
anisotropy_g256_df = anisotropy_within_model_df[anisotropy_within_model_df["name"] == "Gemstone-256x23"]
dim_reduction_g256_df = dim_reduction_within_model_df[dim_reduction_within_model_df["name"] == "Gemstone-256x23"]

simlex_results_g512_df = simlex_results_within_model_df[simlex_results_within_model_df["name"] == "Gemstone-512x11"]
anisotropy_g512_df = anisotropy_within_model_df[anisotropy_within_model_df["name"] == "Gemstone-512x11"]
dim_reduction_g512_df = dim_reduction_within_model_df[dim_reduction_within_model_df["name"] == "Gemstone-512x11"]

# RQ1: Effects of Standardization on CP-Data vs. Huge-Data Models

## 4.1.1. Across models: 
Models trained on small data do not benefit from standardization, but models trained on huge data do.

In [ ]:
size_order = {"10M": 0, "100M": 1, "Huge": 2}
simlex_results_across_model_df["size_rank"] = simlex_results_across_model_df["data_size"].map(size_order)
model_order = (
    simlex_results_across_model_df
    .sort_values(["size_rank", "name"])
    ["name"]
    .unique()
    .tolist()
)
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(
    data=simlex_results_across_model_df,
    x="name",
    y="spearman",
    hue="standardization",  
    order=model_order,
    ax=ax
)
group_index = 0
group_indices = []
for size_rank, group_df in simlex_results_across_model_df.groupby("size_rank"):
    group_name = group_df["data_size"].iloc[0]
    group_size = len(group_df["name"].unique())
    group_indices.append((group_name, group_index, group_index + group_size))
    group_index += group_size

colors = ['grey', 'white']
for i, (label, start, end) in enumerate(group_indices):
    ax.axvspan(start - 0.5, end - 0.5, facecolor=colors[i % 2], alpha=0.4, zorder=0)
    ax.text((start + end - 1) / 2, ax.get_ylim()[1] * 0.97,
            label, ha='center', va='top', fontsize=9, fontweight='bold')

plt.title("Average Spearman Correlation by Standardization Condition for Each Model")
plt.xlabel("Model Class")
plt.xticks(rotation=45, ha='right')
plt.ylabel("Average Spearman Correlation")
plt.legend(title="Standardization Condition")
plt.show()


Does standardization hurt small models, or does it just not help?

In [ ]:
delta_across_models_df = compute_delta_spearman(
    simlex_results_across_model_df,
    ["name", "layer", "data_size"],
)
delta_across_models_df.head()

In [ ]:
# Small models only:
small_models_df = delta_across_models_df[delta_across_models_df["data_size"].isin(["10M", "100M"])]

In [ ]:
# Per layer, not averaged across layers:
t_stat, p_two_tailed = stats.ttest_1samp(small_models_df['delta_spearman'], popmean=0)
print("t-test for small models, per layer (not averaged across layers):")
print(f"t-statistic: {t_stat}, p-value: {p_two_tailed}")

In [ ]:
# Averaged across layers:
small_models_avg_df = small_models_df.groupby("name")["delta_spearman"].mean().reset_index()
t_stat_avg, p_two_tailed_avg = stats.ttest_1samp(small_models_avg_df['delta_spearman'], popmean=0)
print("\nt-test for small models, averaged across layers:")
print(f"t-statistic: {t_stat_avg:.4f}, p-value: {p_two_tailed_avg:.4f}")

In [ ]:
print("Mean delta Spearman correlation for small models:")
print(f"{small_models_avg_df['delta_spearman'].mean():.4f}")

In [ ]:
# Delta spearman for models trained on huge:
huge_models_df = delta_across_models_df[delta_across_models_df["data_size"] == "Huge"]
print("\nDelta Spearman correlation for models trained on huge:")
print(f"{huge_models_df['delta_spearman'].mean():.4f}")

In [ ]:
t_stat, p_two_tailed = stats.ttest_1samp(huge_models_df['delta_spearman'], popmean=0)
print("t-test for huge models, per layer (not averaged across layers):")
print(f"t-statistic: {t_stat}, p-value: {p_two_tailed}")

In [ ]:
huge_models_avg_df = huge_models_df.groupby("name")["delta_spearman"].mean().reset_index()
print("\nDelta Spearman correlation for models trained on huge (averaged across layers):")
print(f"{huge_models_avg_df['delta_spearman'].mean():.4f}")
t_stat_avg_huge, p_two_tailed_avg_huge = stats.ttest_1samp(huge_models_avg_df['delta_spearman'], popmean=0)
print("\nt-test for huge models, averaged across layers:")
print(f"t-statistic: {t_stat_avg_huge:.4f}, p-value: {p_two_tailed_avg_huge:.4f}")

In [ ]:
# Summary bar graph: mean delta spearman for small models vs. mean delta spearman for large
small_models_avg_df['Data Scale'] = 'CP (10M-100M words)'
huge_models_avg_df['Data Scale'] = 'Huge (100B+ tokens)'

combined_df = pd.concat([small_models_avg_df, huge_models_avg_df], ignore_index=True)

fig, ax = plt.subplots(figsize=(5, 4))

sns.barplot(
    data=combined_df,
    x='Data Scale',
    y='delta_spearman',
    hue='Data Scale',
    palette={
        'CP (10M-100M words)': 'steelblue',
        'Huge (100B+ tokens)': 'steelblue'
    },
    capsize=0.1,
    err_kws={'linewidth': 1.5},
    ax=ax,
    legend=False,
    width=0.4
)

ax.axhline(0, color='black', linewidth=1.0)
ax.set_xlabel('')
ax.set_ylabel('Mean Δρ')
ax.set_title('Effect of Standardization by Training Data Scale')

plt.tight_layout()
plt.show()


In [ ]:
huge_models_avg_df

## 4.1.2: Within Models
Standardization hurts lexical similarity performance early in training, but as the models see more data, standardization begins to help.

In [ ]:
# Get the delta-spearman for each checkpoint row.
delta_within_model_df = compute_delta_spearman(
    simlex_results_within_model_df, ["name", "layer", "step", "tokens_seen"]
)

In [ ]:
plot_delta_spearman_by_tokens_seen(delta_within_model_df[delta_within_model_df["name"] == "Pythia-14M"], "Average Δ Spearman Correlation for Pythia-14m Checkpoints")
plot_delta_spearman_by_tokens_seen(delta_within_model_df[delta_within_model_df["name"] == "Pythia-70M"], "Average Δ Spearman Correlation for Pythia-70m Checkpoints")
plot_delta_spearman_by_tokens_seen(delta_within_model_df[delta_within_model_df["name"] == "Gemstone-256x23"], "Average Δ Spearman Correlation for Gemstone-256x23 Checkpoints")
plot_delta_spearman_by_tokens_seen(delta_within_model_df[delta_within_model_df["name"] == "Gemstone-512x11"], "Average Δ Spearman Correlation for Gemstone-512x11 Checkpoints")


In [ ]:
# Use checkpoints with <10B tokens seen as "low end" (cognitively plausible scale)
# and checkpoints with >=100B tokens seen as "high end" (huge scale) for the analysis.
low_end_delta_within_model_df = delta_within_model_df[delta_within_model_df["tokens_seen"] < 10_000_000_000]
high_end_delta_within_model_df = delta_within_model_df[delta_within_model_df["tokens_seen"] >= 100_000_000_000]
early_means = low_end_delta_within_model_df.groupby("name")["delta_spearman"].mean()
late_means = high_end_delta_within_model_df.groupby("name")["delta_spearman"].mean()

In [ ]:
print("Average delta-spearman for checkpoints with <10B tokens seen:")
print(early_means)
print("Average delta-spearman for checkpoints with >100B tokens seen:")
print(late_means)

Note that for all models, delta spearman is negative for early checkpoints and positive for late checkpoints. 

In [ ]:
# align them
early_means = early_means.loc[late_means.index]  # make sure order matches

# parametric
t_stat, p_val = ttest_rel(late_means, early_means)
print(f"Paired t-test: t={t_stat:.4f}, p={p_val:.4f}")

The paired, two-tailed t-test indicates that the difference in standardization effectiveness is significant.
Given the small number of models (n= 4), we additionally verify using a mixed linear model regression model on all layers of all
CP- and huge-data checkpoints with model name as a random effect.

In [ ]:
low_end_delta_within_model_df["early_late"] = "Early (<10B tokens)"
high_end_delta_within_model_df["early_late"] = "Late (>100B tokens)"
high_and_low_df = pd.concat([low_end_delta_within_model_df, high_end_delta_within_model_df], ignore_index=True)
model =smf.mixedlm("delta_spearman ~ early_late", data=high_and_low_df, groups=high_and_low_df["name"]).fit()
print(f"MixedLM regression results for early vs. late checkpoints (Per layer):")
print(model.summary())

high_and_low_means_df = high_and_low_df.groupby(["name", "step", "early_late"])["delta_spearman"].mean().reset_index()
model =smf.mixedlm("delta_spearman ~ early_late", data=high_and_low_means_df, groups=high_and_low_means_df["name"]).fit()
print(f"MixedLM regression results for early vs. late checkpoints (Layer averages):")
print(model.summary())

# RQ2: Does rogue dimensionality explain the observed difference?

Timkey and van Schijndel (2021) observed that in the LLMs they studied, anisotropy was not a global property of the embedding space, but was driven by a small number of what they termed "rogue dimensions." They calculated the contribution of each dimension to cosine similarity for each pair of tokens, enabling them to see how much each dimension contributed to a high anisotropy score. They then calculated the proportion of variance in cosine similarity of all dimensions that was accounted for by cosine similarity of all but the top $k$ dimensions. 

We follow their approach, computing $r^2$, the square of the Pearson correlation between cosine similarity of the tokens with all their dimensions and the cosine similarity of the tokens with the top $k$ dimensions removed. If removing a dimension had very little impact, $r^2$ will be close to 1. The more a given layer contributes to the variabilty in cosine similarity--that is, the more "rogue" it is--the closer to 0 $r^2$ will be when that dimension is removed. In other words, high values mean the dimensions are not very rogue. 

Timkey and van Schijndel were motivated by these rogue dimensions to apply standardization. We therefore ask whether $r^2$ for a given model and layer predicts the usefulness of standardization.


## Across Models

In [ ]:
# Merge the dim reduction data with the delta spearman across models data.
r2_merged_across_model_df = pd.merge(delta_across_models_df, dim_reduction_across_model_df, left_on=["name", "layer", "data_size"], right_on=["name", "layer", "data_size"])
r2_merged_across_model_df.head()

### Does rogue dimensionality correlate with delta spearman?

In [ ]:
for k, group_df in r2_merged_across_model_df.groupby("k"):
    report_spearman(group_df, label=f"({k})", indep_var="r2")

In [ ]:
# What if we do the mean over all layers for each model, instead of per-layer? 
for k, group_df in r2_merged_across_model_df.groupby("k"):
    mean_df = group_df.groupby("name")[["r2", "delta_spearman"]].mean().reset_index()
    report_spearman(mean_df, label=f"({k})", indep_var="r2")

Spearman shows a negative correlation. That is, as r2 increases (i.e., rogue dimensionality decreases), delta spearman decreases. This is consistent with the original paper. 

In [ ]:
for k, group_df in r2_merged_across_model_df.groupby("k"):
    model =smf.mixedlm("delta_spearman ~ r2", data=group_df, groups=group_df["name"]).fit()
    print(f"MixedLM regression results for {k}:")
    print(model.summary())

In [ ]:
r2_merged_across_model_df.columns

For k=3 and k=5, the direction of correlation and the siginficance of results hold with a mixed linear model regression.
However, for k=1, p=0.378. We therefore cannot reject the null hypothesis for k=1. We cannot conclude that the variance accounted for by a single rogue dimension is correlated with how much standardization helps.

### Relationship of data scale and rogue dimensionality

In [ ]:
# Let's do a histogram of the r^2 values for each k, colored by data scale.
for k, group_df in r2_merged_across_model_df.groupby("k"):
    group_df["size_rank"] = group_df["data_size"].map(size_order)
    sns.histplot(data=group_df.sort_values("size_rank"), x="r2", hue="data_size", multiple="stack", binwidth=0.02)
    plt.title(f"Distribution of R² Values for Each Data Scale ({k})")
    plt.xlabel("R²")
    plt.ylabel("Count")
    plt.show()

In [ ]:
for k, group_df in r2_merged_across_model_df.groupby("k"):
    # Print the means of the three groups.
    print(f"Mean R² for each data scale ({k}):")
    mean_r2_df = group_df.groupby("data_size")["r2"].mean().reset_index()
    for row in mean_r2_df.sort_values("data_size", key=lambda x: x.map(size_order)).itertuples():
        print(f"{row.data_size}: {row.r2:.3f}")

### Mediation Analysis

In [ ]:
r2_merged_across_model_df["tokens_seen"] = None
add_tokens_seen_estimates(r2_merged_across_model_df)
r2_merged_across_model_df["log_tokens_seen"] = np.log10(r2_merged_across_model_df["alt_tokens_seen"].astype(float))



In [ ]:
# Across all models, all layers
for k, group_df in r2_merged_across_model_df.groupby("k"):
    model = pingouin.mediation_analysis(data=group_df, x="log_tokens_seen", m="r2", y="delta_spearman", alpha=0.05)
    model.set_index("path", inplace=True)
    print(f"Mediation analysis results for {k}:")
    print(model)

    total_coeff = model.loc["Total", "coef"]
    direct_coeff = model.loc["Direct", "coef"]
    indirect_coeff = model.loc["Indirect", "coef"]

    if model.loc["Indirect", "sig"] == "Yes":
        print(f"Significant mediation effect detected for {k} (p={model.loc['Indirect', 'pval']:.4f}).")
        percent_mediated = (indirect_coeff / total_coeff) * 100 if total_coeff != 0 else np.nan
        print(f"Percentage of total effect mediated by R² for {k}: {percent_mediated:.2f}%")
        print()
    else:
        print(f"No significant mediation effect detected for {k} (p={model.loc['Indirect', 'pval']:.4f}).")
        print()

In [ ]:
# Across all models, averaged across layers (reported in paper)
for k, group_df in r2_merged_across_model_df.groupby("k"):
    mean_df = group_df.groupby("name")[["r2", "delta_spearman", "log_tokens_seen"]].mean().reset_index()
    model = pingouin.mediation_analysis(data=mean_df, x="log_tokens_seen", m="r2", y="delta_spearman", alpha=0.05)
    model.set_index("path", inplace=True)
    print(f"Mediation analysis results for {k} (average over layers):")
    print(model)

    total_coeff = model.loc["Total", "coef"]
    direct_coeff = model.loc["Direct", "coef"]
    indirect_coeff = model.loc["Indirect", "coef"]

    if model.loc["Indirect", "sig"] == "Yes":
        print(f"Significant mediation effect detected for {k} (p={model.loc['Indirect', 'pval']:.4f}).")
        percent_mediated = (indirect_coeff / total_coeff) * 100 if total_coeff != 0 else np.nan
        print(f"Percentage of total effect mediated by R² for {k}: {percent_mediated:.2f}%")
        print()
    else:
        print(f"No significant mediation effect detected for {k} (p={model.loc['Indirect', 'pval']:.4f}).")
        print()

## Within Models
### Mediation Analysis

In [ ]:
delta_within_models_df = compute_delta_spearman(
    simlex_results_within_model_df,
    ["name", "layer", "step"],
)
# Merge the dim reduction data with delta spearman for each checkpoint row in the within-model analysis.
r2_merged_within_model_df = pd.merge(delta_within_models_df, dim_reduction_within_model_df, left_on=["name", "layer", "step"], right_on=["name", "layer", "step"])
r2_merged_within_model_df["log_tokens_seen"] = np.log10(r2_merged_within_model_df["tokens_seen"].astype(float))

In [ ]:
# Mediation analysis for each model, within each k, across all layers
for k, group_df in r2_merged_within_model_df.groupby("k"):
    for model, model_df in group_df.groupby("name"):
        print(f"Mediation analysis for {model} ({k}):")
        mediation = pingouin.mediation_analysis(data=model_df, x="log_tokens_seen", m="r2", y="delta_spearman", alpha=0.05)
        mediation.set_index("path", inplace=True)
        print(mediation)
        if mediation.loc["Indirect", "sig"] == "Yes":
            total_coeff = mediation.loc["Total", "coef"]
            indirect_coeff = mediation.loc["Indirect", "coef"]
            percent_mediated = (indirect_coeff / total_coeff) * 100 if total_coeff != 0 else np.nan
            print(f"Significant mediation effect detected for {model} ({k}) (p={mediation.loc['Indirect', 'pval']:.4f}).")
            print(f"Percentage of total effect mediated by R² for {model} ({k}): {percent_mediated:.2f}%")
        else:
            print(f"No significant mediation effect detected for {model} ({k}) (p={mediation.loc['Indirect', 'pval']:.4f}).")

        print()

In [ ]:
# Mediation analysis for each model, within each k, averaged across layers (reported in paper)
for k, group_df in r2_merged_within_model_df.groupby("k"):
    for model, model_df in group_df.groupby("name"):
        print(f"Mediation analysis for {model} ({k}) Averaged over layers:")
        mean_df = model_df.groupby("step")[["r2", "delta_spearman", "log_tokens_seen"]].mean().reset_index()
        mediation = pingouin.mediation_analysis(data=mean_df, x="log_tokens_seen", m="r2", y="delta_spearman", alpha=0.05)
        mediation.set_index("path", inplace=True)
        print(mediation)
        if mediation.loc["Indirect", "sig"] == "Yes":
            total_coeff = mediation.loc["Total", "coef"]
            indirect_coeff = mediation.loc["Indirect", "coef"]
            percent_mediated = (indirect_coeff / total_coeff) * 100 if total_coeff != 0 else np.nan
            print(f"Significant mediation effect detected for {model} ({k}) (p={mediation.loc['Indirect', 'pval']:.4f}).")
            print(f"Percentage of total effect mediated by R² for {model} ({k}): {percent_mediated:.2f}%")
        else:
            print(f"No significant mediation effect detected for {model} ({k}) (p={mediation.loc['Indirect', 'pval']:.4f}).")

        print()

In [ ]:
# Simplified output: Which ones were significant, in which direction?
simple_impact_results = []
for k, group_df in r2_merged_within_model_df.groupby("k"):
    for model, model_df in group_df.groupby("name"):
        mean_df = model_df.groupby("step")[["r2", "delta_spearman", "log_tokens_seen"]].mean().reset_index()
        mediation = pingouin.mediation_analysis(data=mean_df, x="log_tokens_seen", m="r2", y="delta_spearman", alpha=0.05)
        mediation.set_index("path", inplace=True)
        if mediation.loc["Indirect", "sig"] == "Yes":
            total_coeff = mediation.loc["Total", "coef"]
            indirect_coeff = mediation.loc["Indirect", "coef"]
            percent_mediated = (indirect_coeff / total_coeff) * 100 if total_coeff != 0 else np.nan
            res = {
                "model": model,
                "k": k,
                "total_coeff": total_coeff,
                "indirect_coeff": indirect_coeff,
                "percent_mediated": percent_mediated,
                "pval": mediation.loc["Indirect", "pval"],
            }
            simple_impact_results.append(res)
        else:
            res = {
                "model": model,
                "k": k,
                "total_coeff": mediation.loc["Total", "coef"],
                "indirect_coeff": mediation.loc["Indirect", "coef"],
                "percent_mediated": np.nan,
                "pval": mediation.loc["Indirect", "pval"],
            }
            simple_impact_results.append(res)

simple_impact_df = pd.DataFrame(simple_impact_results)
print("Summary of mediation analysis results (R² as mediator):")
print(simple_impact_df)

# RQ3: Does anisotropy explain the observed differences?
We follow Ethayarajh et al. (2019)'s measure of ansisotropy as the average cosine similarity of random pairs of tokens. A score of 1 indicates high anisotropy, while a score of 0 indicates no anisotropy. We measure anisotropy for each layer of each model.
We sample 500k pairs of tokens from a set of Wikipedia sentences; sentences in the training data for our BaselineLM are excluded, but there could be overlap with train sets for other models. 

## Across Models

In [ ]:
merged_across_models_df = pd.merge(delta_across_models_df, anisotropy_across_model_df, left_on=["name", "layer", "data_size"], right_on=["name", "layer", "data_size"])

### Does anisotropy correlate with delta spearman?

In [ ]:
print("Spearman correlation between anisotropy and delta-spearman across all models, all layers:")
report_spearman(merged_across_models_df)

The Spearman r of 0.2964 (p < 0.0001) between anisotropy and delta Spearman indicates a small but statistically significant positive correlation: models/layers with more anisotropy are more likely to benefit from standardization. 

In [ ]:
merged_across_models_mean_df = merged_across_models_df.groupby("name")[["anisotropy", "delta_spearman"]].mean().reset_index()

print("Spearman correlation between anisotropy and delta-spearman across all models, averaged across layers:")
report_spearman(merged_across_models_mean_df)

The correlation appears stronger when we use the average across layers of each model; the significance is reduced (perhaps due to the much smaller n), but still $p<0.05$

In [ ]:
model = smf.mixedlm("delta_spearman ~ anisotropy", data=merged_across_models_df, groups=merged_across_models_df["name"]).fit()
print(model.summary())

The mixed linear model using model name as a random effect to address the possibility that results from layers of the same model may not be independently distributed yields a coefficient of 0.084 (p<0.001).

### Mediation Analysis
We observed that anisotropy is correlated with the effectiveness of standardization. We also observed that training data size predicts the effectiveness of standardization. Does the increasing anisotropy of models trained on large data mediate the effect?

In [ ]:
add_tokens_seen_estimates(merged_across_models_df)
merged_across_models_df["log_tokens_seen"] = np.log10(merged_across_models_df["alt_tokens_seen"].astype(float))

In [ ]:
# Mediation analysis for anisotropy as a mediator between log_tokens_seen and delta_spearman across all models, all layers
ma = pingouin.mediation_analysis(data=merged_across_models_df, x="log_tokens_seen", m="anisotropy", y="delta_spearman", alpha=0.05)
# Make "path" the index
ma.set_index("path", inplace=True)
print(ma)

In [ ]:
# Mediation analysis for anisotropy as a mediator between log_tokens_seen and delta_spearman across all models, averaged across layers
# (reported in paper)
merged_across_models_mean_df = merged_across_models_df.groupby("name")[["anisotropy", "delta_spearman", "log_tokens_seen"]].mean().reset_index()
ma = pingouin.mediation_analysis(data=merged_across_models_mean_df, x="log_tokens_seen", m="anisotropy", y="delta_spearman", alpha=0.05, seed=42)
# Make "path" the index
ma.set_index("path", inplace=True)
print(ma)

## Within Models

In [ ]:
merged_within_models_df = pd.merge(delta_within_models_df, anisotropy_within_model_df, left_on=["name", "layer", "step"], right_on=["name", "layer", "step"])

### Mediation Analyeses

In [ ]:
# Mediation analysis for anisotropy as a mediator between log_tokens_seen and delta_spearman within each model, across all layers
merged_within_models_df["log_tokens_seen"] = np.log10(merged_within_models_df["tokens_seen"])
result_dfs = {}
for model, group_df in merged_within_models_df.groupby("name"):
    print(f"Mediation analysis for {model}:")
    mediation = pingouin.mediation_analysis(data=group_df, x="log_tokens_seen", m="anisotropy", y="delta_spearman", alpha=0.05)
    mediation.set_index("path", inplace=True)
    print(mediation)
    result_dfs[model] = mediation
    if mediation.loc["Indirect", "sig"] == "Yes":
        total_coeff = mediation.loc["Total", "coef"]
        indirect_coeff = mediation.loc["Indirect", "coef"]
        percent_mediated = (indirect_coeff / total_coeff) * 100 if total_coeff != 0 else np.nan
        print(f"Significant mediation effect detected for {model} (p={mediation.loc['Indirect', 'pval']:.4f}).")
        print(f"Percentage of total effect mediated by anisotropy for {model}: {percent_mediated:.2f}%")
    else:
        print(f"No significant mediation effect detected for {model} (p={mediation.loc['Indirect', 'pval']:.4f}).")

    print()

In [ ]:
# Mediation analysis for anisotropy as a mediator between log_tokens_seen and delta_spearman within each model, averaged across layers
# Note: This is the analysis reported in the paper.
# The run reported was unseeded, so the indirect-effect p-values will differ slightly between runs.
# See the next cell for analysis of multiple seeds to check for robustness of the results.
means_result_dfs = {}
for model, group_df in merged_within_models_df.groupby("name"):
    print(f"Mediation analysis for {model}:")
    means_df = group_df.groupby("step")[["anisotropy", "delta_spearman", "log_tokens_seen"]].mean().reset_index()
    mediation = pingouin.mediation_analysis(data=means_df, x="log_tokens_seen", m="anisotropy", y="delta_spearman", alpha=0.05) # Setting seed = 486 will reproduce the exact results from the paper.
    mediation.set_index("path", inplace=True)
    print(mediation)
    means_result_dfs[model] = mediation
    if mediation.loc["Indirect", "sig"] == "Yes":
        total_coeff = mediation.loc["Total", "coef"]
        indirect_coeff = mediation.loc["Indirect", "coef"]
        percent_mediated = (indirect_coeff / total_coeff) * 100 if total_coeff != 0 else np.nan
        print(f"Significant mediation effect detected for {model} (p={mediation.loc['Indirect', 'pval']:.4f}).")
        print(f"Percentage of total effect mediated by anisotropy for {model}: {percent_mediated:.2f}%")
    else:
        print(f"No significant mediation effect detected for {model} (p={mediation.loc['Indirect', 'pval']:.4f}).")


In [ ]:
# Diagnostic: how stable is the Pythia-14M indirect-effect p-value across bootstrap seeds?
p14 = merged_within_models_df[merged_within_models_df["name"] == "Pythia-14M"]
p14_means = p14.groupby("step")[["anisotropy", "delta_spearman", "log_tokens_seen"]].mean().reset_index()
pvals, ci_low, ci_high = [], [], []
for seed in range(20):
    res = pingouin.mediation_analysis(data=p14_means, x="log_tokens_seen", m="anisotropy",
                                        y="delta_spearman", alpha=0.05, seed=seed).set_index("path")
    pvals.append(res.loc["Indirect", "pval"])
    ci_low.append(res.loc["Indirect", "CI2.5"]); ci_high.append(res.loc["Indirect", "CI97.5"])
pvals = np.array(pvals)
print(f"p min={pvals.min():.4f}, median={np.median(pvals):.4f}, max={pvals.max():.4f}, "
        f"share of seeds with p<0.05: {(pvals < 0.05).mean():.2f}, "
        f"share of seeds whose 95% CI excludes 0: {np.mean(np.array(ci_low) > 0):.2f}")

After submission of the camera-ready paper, repeated reruns of the mediation analysis with different seeds suggests that the indirect effect for Pythia-14 is marginally significant according to p value. However, the 95% confidence interval crosses zero in 50% of runs, suggesting that it is *not* significant; see Kristopher J. Preacher and Andrew F. Hayes. 2004. SPSS and SAS procedures for estimating indirect effects in simple mediation models. Behavior Research Methods, Instruments, & Computers, 36(4):717–731, strongly encouraging the use of confidence intervals for mediation analyses.


# Discussion
## A. Does Baseline semantic similarity performance mediate the effect?

In [ ]:
# Combine standardized/not-standardized spearman values for each model/layer into a single row.
standardization_comparison_df = (
    simlex_results_across_model_df
    .pivot_table(index=["name", "layer", "data_size"], columns="standardization", values="spearman")
    .rename(columns={
        "Not Standardized": "not_standardized_spearman",
        "Standardized": "standardized_spearman",
    })
    .reset_index()
)
standardization_comparison_df["delta_spearman"] = (
    standardization_comparison_df["standardized_spearman"]
    - standardization_comparison_df["not_standardized_spearman"]
)

# The pivot drops "step" (it isn't part of the index/values), but add_tokens_seen_estimates
# needs it for the Gemstone "Huge" rows, so bring it back via a lookup.
step_lookup = simlex_results_across_model_df[["name", "layer", "data_size", "step"]].drop_duplicates()
standardization_comparison_df = standardization_comparison_df.merge(
    step_lookup, on=["name", "layer", "data_size"], how="left"
)
standardization_comparison_df["tokens_seen"] = None
add_tokens_seen_estimates(standardization_comparison_df)
standardization_comparison_df.head()


In [ ]:
standardization_comparison_df["alt_tokens_seen"].value_counts(dropna=False)

In [ ]:
standardization_comparison_df["log_tokens_seen"] = np.log10(standardization_comparison_df["alt_tokens_seen"].astype(float))


In [ ]:
ma = pingouin.mediation_analysis(data=standardization_comparison_df, x="log_tokens_seen", m="not_standardized_spearman", y="delta_spearman", alpha=0.05)
# Make "path" the index
ma.set_index("path", inplace=True)
print(ma)

In [ ]:
total_coeff = ma.loc["Total", "coef"]
direct_coeff = ma.loc["Direct", "coef"]
indirect_coeff = ma.loc["Indirect", "coef"]

# Percentage of total effect that is mediated
percent_mediated = (indirect_coeff / total_coeff) * 100 if total_coeff != 0 else np.nan
print(f"Percentage of total effect mediated by baseline spearman: {percent_mediated:.2f}%")

In [ ]:
import statsmodels.api as sm

X = standardization_comparison_df[['log_tokens_seen', 'not_standardized_spearman']]
X = sm.add_constant(X)
y = standardization_comparison_df['delta_spearman']
model = sm.OLS(y, X, missing='drop').fit()
print(model.params)  # coefficient on not_standardized_spearman here = the TRUE b

In [ ]:
print(model.summary())

In [ ]:
print(standardization_comparison_df[['log_tokens_seen', 'not_standardized_spearman']].corr())

In [ ]:
# Calculate VIF:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = standardization_comparison_df[['log_tokens_seen', 'not_standardized_spearman']]
X = sm.add_constant(X)
vif_data = pd.DataFrame()
vif_data["variable"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

# Appendix A: Isolated vs. Contextualized Word Vectors

In [ ]:
# Which is better, isolated or contextualized, in terms of absolute Spearman correlation?
avg_spearman_by_condition = isolated_vs_contextualized_df.groupby("embedding_mode")["spearman"].mean()
print(avg_spearman_by_condition)

# Let's add some other stats: max, min, std by condition.
spearman_stats_by_condition = isolated_vs_contextualized_df.groupby("embedding_mode")["spearman"].agg(["min", "max", "mean", "std"])
print(spearman_stats_by_condition)

In [ ]:
context_spearmans = isolated_vs_contextualized_df[isolated_vs_contextualized_df["embedding_mode"] == "contextualized"]["spearman"]
isolated_spearmans = isolated_vs_contextualized_df[isolated_vs_contextualized_df["embedding_mode"] == "isolated"]["spearman"]
t_stat, p_value = ttest_rel(context_spearmans, isolated_spearmans)
print(f"Paired t-test results: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")